In [4]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,AutoModelForCausalLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


In [2]:
dpo_train_data = [
    {
        "prompt": "Review: This movie is a masterpiece. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: I hated this film. Total waste of time. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    },
    {
        "prompt": "Review: Outstanding acting and brilliant plot. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Terrible acting and boring screenplay. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"

    }
]

dpo_test_data = [
    {
        "prompt": "Review: The movie was mediocre, but the ending was spectacular! Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    }
]

train_dataset = Dataset.from_list(dpo_train_data)
test_dataset = Dataset.from_list(dpo_test_data)
print("DPO dataset initialized!")

DPO dataset initialized!


In [ ]:
model = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model).to(device)

# gpt-2 패딩 조정
tokenizer.pad_token = tokenizer.add_eos_token
model.config.pad_token_id = model.config.eos

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8891.07it/s]
